## nanoGPT

In [2]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
# !wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [3]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(f"len = {len(text)}")

len = 1115394


In [4]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [5]:
itos = {i: s for i, s in enumerate(chars)}
stoi = {s: i for i, s in enumerate(chars)}
def encode(s):
    return [stoi[l] for l in s]

def decode(li):
    return ''.join(itos[i] for i in li)

s = 'Hello'
print(encode(s))
print(decode(encode(s)))

[20, 43, 50, 50, 53]
Hello


In [6]:
import torch

data = torch.tensor(encode(text), dtype=int)

n = round(.9 * len(data))

train_data = data[:n]
val_data = data[n:]


In [47]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8
n_embd = 32

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    Xb = torch.stack([data[i:i+block_size] for i in ix])
    Yb = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return Xb, Yb

Xb, Yb = get_batch('train')
print(Xb.shape)
Xb, Yb

torch.Size([4, 8])


(tensor([[56,  6,  0, 24, 43, 58,  1, 61],
         [39, 47, 51,  1, 58, 46, 39, 58],
         [52, 45,  1, 58, 53,  1, 57, 39],
         [43, 47, 52, 45,  1, 46, 53, 50]]),
 tensor([[ 6,  0, 24, 43, 58,  1, 61, 46],
         [47, 51,  1, 58, 46, 39, 58,  1],
         [45,  1, 58, 53,  1, 57, 39, 63],
         [47, 52, 45,  1, 46, 53, 50, 47]]))

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

# bigram : predict next token based on only the current one.
class Bigram(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embd_table = nn.Embedding(vocab_size, n_embd)
        self.pos_embd_table = nn.Embedding(block_size, n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, x, target=None):
        B, T = x.shape
        tok_emb = self.token_embd_table(x) # (B, T, n_emb)
        pos_emb = self.pos_embd_table(torch.arange(T)) # (T, n_emb)
        x = tok_emb + pos_emb 
        logits = self.lm_head(x)
        if target == None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(-1, C)
            target = target.view(-1)
            loss = F.cross_entropy(logits, target)
            logits = logits.view(B, T, C)
        return logits, loss

    def generate(self, ix, max_new_tokens):
        logits, loss = self(ix)
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)

        # multinomial takes 1D or 2D tensors only
        # always sample from the last dimension.
        # (N,) -> (num_samples,)
        #(B, N) -> (B, num_samples)
        idx_next = torch.multinomial(probs, num_samples=1) 
        idx = torch.cat((ix, idx_next), dim=1)
        return idx

In [49]:
model = Bigram()
torch.manual_seed(1337)

idx = torch.zeros((1, 1), dtype=int)
letters = 100
for _ in range(letters):
    idx = model.generate(idx, letters)
s = decode(idx[0].tolist())
print(s)


CpzlYoSy;czdeYQwEiYjYzrzlGu-?YeaYnijbo
LzBjzeULS-A.U'FisdJzG HsssPUW;,qe$zOAnszhimKzBAkfyfY:gSFRNNBE


In [51]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

batch_size = 32

for steps  in range(10000):
    xb, yb = get_batch('train')

    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if(steps == 0):
        print(f'start loss {loss.item()}')

print(f'final loss: {loss.item()}')

start loss 4.317026615142822
final loss: 2.413017988204956


In [52]:
torch.manual_seed(1337)
idx = torch.zeros((1, 1), dtype=int)
letters = 200
for _ in range(letters):
    idx = model.generate(idx, letters)
s = decode(idx[0].tolist())
print(s)


Cprs an tcede.
YE hin roloundee we anonse ate t, bye wist ic wsostte; bea yonsenimsse se ay g pat ancey mou ber s LI'sl tem'ls tofren gre d, IAS thorvere nonifit deanche
Whatrerath; shan iseress tode 


In [36]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2
x = torch.randint(low=0, high=10, size=(B, T, C), dtype=float)
x[0]


tensor([[5., 7.],
        [2., 0.],
        [5., 3.],
        [5., 0.],
        [4., 0.],
        [2., 0.],
        [7., 6.],
        [0., 8.]], dtype=torch.float64)

In [41]:
new_x = x.clone()
for t in range(T):
    xprev = x[:, :t+1]
    new_x[:, t] = torch.mean(xprev, dim=1)

new_x[0]

tensor([[5.0000, 7.0000],
        [3.5000, 3.5000],
        [4.0000, 3.3333],
        [4.2500, 2.5000],
        [4.2000, 2.0000],
        [3.8333, 1.6667],
        [4.2857, 2.2857],
        [3.7500, 3.0000]], dtype=torch.float64)

In [ ]:
B, T, C = 4, 8, 32
head_size = 16
x = torch.randn(B, T, C)
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)
q = query(x)
v = value(x) # B x T x 16
wei = q @ k.transpose(-2, -1) #* head_size**-.5
# wei = torch.zeros(T, T)
tril = torch.tril(torch.ones((T, T), dtype=float)) 
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, -1)
out = wei @ v


In [188]:
print(x.var())
print(q.var())
print(k.var())
print(wei.var())

tensor(1.0296)
tensor(0.3364, grad_fn=<VarBackward0>)
tensor(0.3412, grad_fn=<VarBackward0>)
tensor(0.0293, grad_fn=<VarBackward0>)
